# 17. Leakage and honesty checks (audit of 10 November)
**Purpose:** show that no information from the future or from the test labels reaches the predictions, the cut-offs or the flags.
**Part A: future perturbation.** Rebuild the features as in notebook 03, fit Model C, then replace every reading from 1 December 2016 onward with random values and recompute the history features. Predictions before 1 December must not change (reported: change of exactly 0; after the cut they change by up to 7.4, so the test can detect dependence). It also lists the features, checks that none is a reading, label or split column, and that the latest training row is before the split date.
**Part B: labels.** Recompute the five cut-offs from scratch (must match `cutoffs.json`), recompute them with random test labels (must be identical), and with random validation labels (may differ, because validation labels are used to leave unusual hours out; this is documented). Then a placebo: shift each building's labeled events to random places inside the test period (200 shifts) and count events found at 1% per building (reported: last hour 5.4% on average, Model C 4.2%, against 38.7% and 35.4% with the real labels).
**Writes:** prints only (copy the output into the report appendix if needed). See master section 20.13.

In [ ]:
from config import *
import numpy as np, pandas as pd
RESULTS = ROOT / "results" / "audit"
RESULTS.mkdir(parents=True, exist_ok=True)
O = str(OUT_DIR) + "/"          # results of notebooks 01-10 (run those first)
def read_any(name):
    """Read name.csv.gz or name.csv or name.parquet from OUT_DIR."""
    for ext in (".csv.gz", ".csv", ".parquet"):
        p = OUT_DIR / (name + ext)
        if p.exists():
            return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
    raise FileNotFoundError(name)

## Part A: future perturbation

In [ ]:
# This notebook re-uses the feature building and model fitting of notebook 03 (cells up to the point where predictions are saved).
# Its own output folder is redirected so nothing of notebook 03's is overwritten.
import json, config
import shutil
config.OUT_DIR = RESULTS / "_scratch"; config.OUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(OUT_DIR / "labeled_building_ids.csv", config.OUT_DIR)   # the only file of notebook 01 that notebook 03 reads
nb03 = json.load(open("03_gepiii_predictors.ipynb"))
ns = {"__name__": "__main__"}
def run_nb03(stop_when):
    for c in nb03["cells"]:
        if c["cell_type"] != "code": continue
        s = "".join(c["source"])
        if stop_when(s): break
        exec(compile(s, "nb03", "exec"), ns)

In [ ]:
run_nb03(lambda s: 'models = {}' in s)   # features and splits only; stop before models are trained here

In [ ]:
df=ns['df'];FE=ns['FEATURES'];fit=ns['fit'];BEST=ns['BEST'];train=ns['train_rows']
print('features of C:',FE['C'],flush=True)
forbidden=[c for c in FE['C'] if c in ('y','meter_reading','label','anomaly','split','in_zero_run')]
print('forbidden columns used as features:',forbidden)
print('latest training timestamp:',train.timestamp.max(),'| TRAIN_END',ns['TRAIN_END'],'| all before TRAIN_END:',bool((train.timestamp<ns['TRAIN_END']).all()))
m=fit(FE['C'],BEST,train)
base=m.predict(df[FE['C']])
CUT=pd.Timestamp('2016-12-01')
d2=df.copy()
# CONTROL: recompute lags from the UNPERTURBED readings and compare with the lags stored by notebook 03
c=df.copy();c['ly']=np.log1p(c['y']);gc=c.groupby('building_id')['ly']
for k in (1,24,168):
    nw=gc.shift(k).to_numpy();old=c[f'lag_{k}'].to_numpy();bad=~((np.isnan(nw)&np.isnan(old))|np.isclose(nw,old,equal_nan=False,atol=1e-6))
    print(f'control lag_{k}: rows that differ from stored lag =',int(bad.sum()))
print('control predictions max diff:',np.nanmax(np.abs(m.predict(c.assign(lag_1=gc.shift(1),lag_24=gc.shift(24),lag_168=gc.shift(168))[FE['C']])-base)))
print('is df sorted by building,time:',bool(df.sort_values(['building_id','timestamp']).index.equals(df.index)),flush=True)
rng=np.random.default_rng(1)
fut=d2.timestamp>=CUT
d2.loc[fut,'y']=rng.uniform(0,5000,fut.sum()).astype('float32')   # destroy every reading from 1 Dec onward
d2['ly']=np.log1p(d2['y']);g=d2.groupby('building_id')['ly']
d2['lag_1']=g.shift(1);d2['lag_24']=g.shift(24);d2['lag_168']=g.shift(168)
new=m.predict(d2[FE['C']])
past=(d2.timestamp<CUT).to_numpy()
diff_past=np.nanmax(np.abs(new[past]-base[past]))
diff_fut=np.nanmax(np.abs(new[~past]-base[~past]))
print('max change in predictions BEFORE the cut (must be 0):',diff_past)
print('max change in predictions AFTER the cut (must be >0, shows the test can detect dependence):',diff_fut)
# 1-hour-ahead strictness: prediction at the first hour of CUT must not change (uses only readings before CUT)
first=(d2.timestamp==CUT).to_numpy()
print('max change at exactly the cut hour (uses only past readings; must be 0):',np.nanmax(np.abs(new[first]-base[first])))

## Part B: labels, cut-offs and placebo

In [ ]:
T0=pd.Timestamp('2016-01-01');TRAIN_END=pd.Timestamp('2016-09-13');VAL_END=pd.Timestamp('2016-11-07')
pred=read_any('predictions');pred['timestamp']=pd.to_datetime(pred.timestamp);lab=read_any('lead_labels')[['building_id','timestamp','label','in_zero_run']];lab['timestamp']=pd.to_datetime(lab.timestamp)
info=pd.read_csv(O+'labeled_buildings_info.csv')[['building_id','site_id','primary_use']]
def cutoffs(labels):
    df=pred.merge(labels,on=['building_id','timestamp'],how='left').merge(info,on='building_id',how='left')
    df['label']=df.label.fillna(0).astype(int);df['in_zero_run']=df.in_zero_run.fillna(False).astype(bool)
    df['split']=np.where(df.timestamp<TRAIN_END,'train',np.where(df.timestamp<VAL_END,'val','test'))
    N=['last_hour','typical','A','B','C'];tf=lambda x:np.log1p(np.clip(x,0,None))
    df['valid']=df.meter_reading.notna()&~df.in_zero_run&df[['pred_'+n for n in N]].notna().all(axis=1)
    for n in N: df['err_'+n]=tf(df.meter_reading)-tf(df['pred_'+n])
    cv=df[(df.split=='val')&df.valid&(df.label==0)];ids=pd.Index(sorted(df.building_id.unique()))
    mad=lambda s:(s-s.median()).abs().median()*1.4826
    stats=pd.DataFrame(index=ids);stats['n']=cv.groupby('building_id').size().reindex(ids).fillna(0).astype(int)
    for n in N:
        g=cv.groupby('building_id')['err_'+n];stats['med_'+n]=g.median().reindex(ids);stats['scale_'+n]=np.maximum(g.apply(mad).reindex(ids),0.05)
    stats=stats.join(info.set_index('building_id')[['site_id','primary_use']]);b=stats.n<500;ok=stats[~b]
    for c in [c for c in stats if c.startswith(('med_','scale_'))]:
        bg=ok.groupby(['site_id','primary_use'])[c].median();bs=ok.groupby('site_id')[c].median()
        key=pd.MultiIndex.from_frame(stats[['site_id','primary_use']]);fill=pd.Series(bg.reindex(key).values,index=stats.index).fillna(stats.site_id.map(bs)).fillna(ok[c].median())
        stats.loc[b,c]=fill[b]
    v=df[(df.split=='val')&df.valid].merge(stats[[c for c in stats if c.startswith(('med_','scale_'))]],left_on='building_id',right_index=True)
    cut={}
    for n in N:
        sc=((v['err_'+n]-v['med_'+n]).abs()/v['scale_'+n]).where(v.label==0).dropna()
        cut[n]=float(np.quantile(sc,0.995))
    return cut
ref=json.load(open(O+'cutoffs.json'))['cutoffs'];ref['last_hour']=ref.pop('last_hour')
c0=cutoffs(lab)
print('recomputed cut-offs   :',{k:round(v,4) for k,v in c0.items()})
print('reported cut-offs     :',{k:round(v,4) for k,v in ref.items()})
print('max abs difference    :',max(abs(c0[k]-ref[k]) for k in c0))
lab2=lab.copy();rng=np.random.default_rng(5)
t=lab2.timestamp>=VAL_END;lab2.loc[t,'label']=rng.integers(0,2,t.sum())   # random test labels
c1=cutoffs(lab2)
print('with RANDOM test labels: max abs difference to recomputed =',max(abs(c1[k]-c0[k]) for k in c0),'(must be 0)')
lab3=lab.copy();v=(lab3.timestamp>=TRAIN_END)&(lab3.timestamp<VAL_END);lab3.loc[v,'label']=rng.integers(0,2,v.sum())
c2=cutoffs(lab3)
print('with RANDOM validation labels: max abs difference =',round(max(abs(c2[k]-c0[k]) for k in c0),4),'(may differ: validation labels are used to exclude anomalous hours, as documented)')
# placebo
f=read_any('flags_ext');f['timestamp']=pd.to_datetime(f.timestamp);f=f[f.split=='test'].sort_values(['building_id','timestamp']).reset_index(drop=True)
f['t']=(f.timestamp-T0)//pd.Timedelta(hours=1);tmin,tmax=int(f.t.min()),int(f.t.max())
ev=pd.read_csv(O+'lead_events.csv',parse_dates=['start','end']);ev['split']=ev.split.replace({'validation':'val'})
ev=ev[(ev.split=='test')&(ev.zero_share<=0.5)&ev.building_id.isin(f.building_id.unique())].copy()
ev['s']=(ev.start-T0)//pd.Timedelta(hours=1);ev['e']=(ev.end-T0)//pd.Timedelta(hours=1)
vt={b:g.t.to_numpy() for b,g in f[f.valid].groupby('building_id')}
cnt=lambda b,s,e:0 if vt.get(b) is None else int(np.searchsorted(vt[b],e,'right')-np.searchsorted(vt[b],s,'left'))
ev=ev[[cnt(b,s,e)>0 for b,s,e in zip(ev.building_id,ev.s,ev.e)]].reset_index(drop=True)
def flags(m):
    s=f['score_'+m].where(f.valid);p=s.groupby(f.building_id).rank(pct=True,method='first');return (p>0.99).fillna(False).to_numpy()
FL={m:{b:g.t.to_numpy() for b,g in f[flags(m)].groupby('building_id')} for m in ['last_hour','C']}
def rec(m,S,E,B):
    T=FL[m];return np.mean([(b in T) and (np.searchsorted(T[b],e,'right')>np.searchsorted(T[b],s,'left')) for b,s,e in zip(B,S,E)])
B=ev.building_id.to_numpy();S=ev.s.to_numpy();E=ev.e.to_numpy()
print('REAL labels   : last hour %.1f%%, Model C %.1f%% of %d events'%(100*rec('last_hour',S,E,B),100*rec('C',S,E,B),len(ev)))
rng=np.random.default_rng(11);res={'last_hour':[],'C':[]}
span=tmax-tmin+1
for k in range(200):
    off=rng.integers(200,span-200,len(ev))  # random shift (hours), wrapped inside the test period
    S2=tmin+((S-tmin+off)%span);E2=S2+(E-S)
    for m in res: res[m].append(rec(m,S2,E2,B))
for m in res: print('PLACEBO %-9s: mean %.1f%%  max %.1f%%  (200 random shifts)'%(m,100*np.mean(res[m]),100*np.max(res[m])))